# Does DeepSeek grant-detail extraction improve ranking?

Compares the **baseline semantic ranking** with the **same semantic ranking plus validated DeepSeek ranking fields**
(`funding_type`, `target_applicants`, `focus_area`, `geography`), on two datasets:

1. **Tracker** — the 37-grant workbook, one applicant profile, evaluation date 2026-10-28, Strong/Excellent labels.
2. **Synthetic** — 150 generated grants × 200 generated profiles (reference date 2026-10-01), rule-generated `match` labels.

Held constant within each dataset: grants, profiles, dates, eligibility rules, embedder (`all-MiniLM-L6-v2`),
chunk weight, the 0.20 similarity threshold, and top 5.

**Extractor v2** (`EXTRACTION_VERSION = "2"`): a value that is not an exact substring of *its own* workbook field
(e.g. `target_applicants` must come from `core_requirements`) is set to `null` and its reason recorded, instead of
discarding the whole grant. Missing keys become `null`; extra keys are ignored and recorded.

**Caveats.** Tracker labels are spreadsheet-fit ratings, not independently verified suitability, and one profile
moves Precision@5 in 20-point steps. Synthetic labels come from the same generator rules as the checker; they test
the method on many profiles but are not real-world accuracy evidence.

Run on SCU Wave with `sbatch notebooks/wave_notebook.sbatch notebooks/deepseek_ranking_evaluation.ipynb`.
Validated extractions are cached in `.cache/deepseek_grant_details.json`; responses with no usable JSON are kept,
with the raw output, in `.cache/deepseek_grant_failures.json`. `DEEPSEEK_RETRY_FAILURES=1` retries those.

In [1]:
import json
import os
import platform
import socket
import sys
import time
from datetime import date, datetime, timezone
from pathlib import Path

# Run from the repo root or from notebooks/.
ROOT = Path.cwd() if (Path.cwd() / "src").is_dir() else Path.cwd().parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 200)

from collections import Counter

from src.ardook_extra_synthetic_data import (
    DB_PATH as SYN_DB, REFERENCE_DATE as SYN_DATE, load_checker as load_syn_checker,
    load_grants as load_syn_grants, load_matches as load_syn_matches, load_profiles as load_syn_profiles,
)
from src.deepseek_extractor import (
    DEFAULT_MODEL, EXTRACTED_FIELDS, EXTRACTION_VERSION, RANKING_FIELDS,
    DeepSeekGrantExtractor, GrantDetailExtractionError, grant_source_text,
)
from src.evaluate_relevance import K, KeywordEmbedder, MemoryStore
from src.ingest import load_grants
from src.pipeline import DEFAULT_CHUNK_WEIGHT, DEFAULT_MIN_SIMILARITY, OrganizationProfile, Pipeline, SemanticEncoder
from src.real_eligibility import load_checker

TRACKER_DATE = date(2026, 10, 28)
TRACKER_PROFILE = OrganizationProfile(
    "Ardook is an Armenia-based social enterprise that provides flexible, work-from-home laundry jobs "
    "and training to single mothers caring for children with disabilities. It offers residential and "
    "commercial laundry services, helping mothers earn income while balancing caregiving.",
    {"region": "Armenia", "entity_type": "social enterprise"},
)

MODEL_NAME = DEFAULT_MODEL
CACHE_PATH = ROOT / ".cache" / "deepseek_grant_details.json"
FAILURE_PATH = ROOT / ".cache" / "deepseek_grant_failures.json"
RESULTS_PATH = ROOT / "notebooks" / "results" / "deepseek_ranking_evaluation.json"
RUN_DEEPSEEK = os.environ.get("DEEPSEEK_RUN", "1") == "1"  # 0 = only use cached results
RETRY_FAILURES = os.environ.get("DEEPSEEK_RETRY_FAILURES", "0") == "1"
assert K == 5 and DEFAULT_MIN_SIMILARITY == 0.20, (K, DEFAULT_MIN_SIMILARITY)
print(f"Repo: {ROOT}\nModel: {MODEL_NAME} | extractor v{EXTRACTION_VERSION}\n"
      f"Top-k: {K} | threshold: {DEFAULT_MIN_SIMILARITY} | chunk weight: {DEFAULT_CHUNK_WEIGHT}")

Repo: /WAVE/users2/unix/gkakar/ardook-deepseek
Model: deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B | extractor v2
Top-k: 5 | threshold: 0.2 | chunk weight: 0.5


## 1. Execution environment

In [ ]:
import torch
import transformers

if torch.cuda.is_available():
    DEVICE, DTYPE = torch.device("cuda"), torch.float16
    device_name = torch.cuda.get_device_name(0)
else:
    # Never MPS: generation crashed on the Mac MPS backend.
    DEVICE, DTYPE = torch.device("cpu"), torch.float32
    device_name = platform.processor() or "cpu"

ENVIRONMENT = {
    "hostname": socket.gethostname(),
    "slurm_job_id": os.environ.get("SLURM_JOB_ID"),
    "slurm_partition": os.environ.get("SLURM_JOB_PARTITION"),
    "cuda_visible_devices": os.environ.get("CUDA_VISIBLE_DEVICES"),
    "python": platform.python_version(),
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "cuda_available": torch.cuda.is_available(),
    "gpu_count": torch.cuda.device_count(),
    "device": str(DEVICE),
    "device_name": device_name,
    "dtype": str(DTYPE),
    "started_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
}
for key, value in ENVIRONMENT.items():
    print(f"{key:>22}: {value}")
if not torch.cuda.is_available():
    print("\.nWARNING: no CUDA GPU visible; model inference will run on CPU and may be very slow.")

              hostname: hub01
          slurm_job_id: 2711918
       slurm_partition: hub
  cuda_visible_devices: 0
                python: 3.12.13
                 torch: 2.11.0+cu130
          transformers: 4.57.6
        cuda_available: True
             gpu_count: 1
                device: cuda
           device_name: NVIDIA RTX PRO 6000 Blackwell Server Edition
                 dtype: torch.float16
           started_utc: 2026-11-11T02:38:02+00:00


## 2. Data

In [3]:
tracker_grants = load_grants()
tracker_checker = load_checker(tracker_grants)
tracker_records = tracker_grants.to_dict("records")
assert len(tracker_records) == 37, len(tracker_records)
tracker_live = [r for r in tracker_records
                if not any(c.status == "fail" for c in tracker_checker.check(r, TRACKER_PROFILE, TRACKER_DATE))]
tracker_labels = {int(r["source_row"]): str(r["fit"]) for r in tracker_live}
tracker_relevant = {int(r["source_row"]) for r in tracker_live if str(r["fit"]).strip().casefold() in {"strong", "excellent"}}

syn_grants = load_syn_grants(SYN_DB)
syn_checker = load_syn_checker(SYN_DB)
syn_records = syn_grants.to_dict("records")
syn_profiles = load_syn_profiles(SYN_DB)
syn_matches = load_syn_matches(SYN_DB)
syn_labels = {int(pid): dict(zip(g.source_row.astype(int), g.label)) for pid, g in syn_matches.groupby("profile_id")}

print(f"Tracker: {len(tracker_records)} grants, {len(tracker_live)} pass known rules on {TRACKER_DATE}, "
      f"{len(tracker_relevant)} Strong/Excellent")
print(f"Synthetic: {len(syn_records)} grants, {len(syn_profiles)} profiles, "
      f"{sum(l == 'match' for d in syn_labels.values() for l in d.values())} match labels")

Tracker: 37 grants, 16 pass known rules on 2026-10-28, 8 Strong/Excellent
Synthetic: 150 grants, 200 profiles, 1401 match labels


## 3. DeepSeek extraction (tracker 37 + synthetic 150 grants)

A grant counts as an extraction *failure* only if the response has no usable JSON object. Otherwise every field is
validated separately; rejected fields become `null` and their reasons are shown below.

In [4]:
extractor = DeepSeekGrantExtractor(MODEL_NAME, cache_path=CACHE_PATH)
failures = json.loads(FAILURE_PATH.read_text(encoding="utf-8")) if FAILURE_PATH.exists() else {}
all_sources = [grant_source_text(r) for r in tracker_records + syn_records]
pending = [s for s in all_sources
           if extractor._cache_key(s) not in extractor.cache
           and (RETRY_FAILURES or extractor._cache_key(s) not in failures)]
print(f"Grant texts: {len(all_sources)} | cached: {len(all_sources) - len(pending)} (incl. recorded failures) | "
      f"need inference: {len(pending)}")
if pending and RUN_DEEPSEEK:
    from transformers import AutoModelForCausalLM, AutoTokenizer
    t0 = time.perf_counter()
    extractor.tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    extractor.model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=DTYPE).to(DEVICE).eval()
    extractor.device = DEVICE
    print(f"Loaded {MODEL_NAME} on {DEVICE} ({DTYPE}) in {time.perf_counter() - t0:.1f}s")

Grant texts: 187 | cached: 0 (incl. recorded failures) | need inference: 187


Loaded deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B on cuda (torch.float16) in 1.4s


In [5]:
def save_failures():
    FAILURE_PATH.parent.mkdir(parents=True, exist_ok=True)
    FAILURE_PATH.write_text(json.dumps(failures, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")


def run_extraction(dataset, records):
    rows, details_by_row = [], {}
    for record in records:
        row, source = int(record["source_row"]), grant_source_text(record)
        key = extractor._cache_key(source)
        entry = {"dataset": dataset, "source_row": row, "program": record["program"], "seconds": None, "error": None}
        if key in failures and key not in extractor.cache and not RETRY_FAILURES:
            entry.update(status="failed (recorded)", error=failures[key]["error"])
        elif key not in extractor.cache and not RUN_DEEPSEEK:
            entry.update(status="not run", error="DEEPSEEK_RUN=0 and no cached result")
        else:
            cached = key in extractor.cache
            t0 = time.perf_counter()
            try:
                details, rejected = extractor.extract_with_report(source)
            except GrantDetailExtractionError as error:
                raw = extractor.last_response or ""
                failures[key] = {"dataset": dataset, "source_row": row, "program": record["program"],
                                 "model": MODEL_NAME, "error": str(error), "raw_response": raw,
                                 "response_tokens": len(extractor.tokenizer(raw).input_ids),
                                 "has_think_close": "</think>" in raw}
                save_failures()
                entry.update(status="failed", error=str(error))
            except Exception as error:  # e.g. CUDA OOM: report, do not record as a deterministic failure
                entry.update(status="error", error=f"{type(error).__name__}: {error}")
            else:
                if key in failures:
                    failures.pop(key)
                    save_failures()
                details_by_row[row] = details
                entry.update(status="ok (cached)" if cached else "ok",
                             accepted=[f for f in EXTRACTED_FIELDS if details[f] is not None],
                             rejected=rejected)
            if not cached:
                entry["seconds"] = round(time.perf_counter() - t0, 1)
        rows.append(entry)
    return details_by_row, rows


t0 = time.perf_counter()
tracker_details, tracker_rows = run_extraction("tracker", tracker_records)
syn_details, syn_rows = run_extraction("synthetic", syn_records)
extraction_rows = tracker_rows + syn_rows
print(f"Extraction finished in {time.perf_counter() - t0:.0f}s; model calls this run: {extractor.model_calls}")

extraction_df = pd.DataFrame(extraction_rows)
extraction_df.groupby(["dataset", "status"]).size().unstack(fill_value=0)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


... (180 outputs left out)

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


Extraction finished in 758s; model calls this run: 187


status,failed,ok
dataset,,
synthetic,2,148
tracker,0,37


In [6]:
# Field-level outcome: accepted / null (model said null) / rejected, per dataset.
field_rows = []
for dataset, group in extraction_df[extraction_df["status"].str.startswith("ok")].groupby("dataset"):
    for field in EXTRACTED_FIELDS:
        accepted = sum(field in a for a in group["accepted"])
        rejected = sum(field in r for r in group["rejected"])
        field_rows.append({"dataset": dataset, "field": field, "accepted": accepted, "rejected": rejected,
                           "null": len(group) - accepted - rejected,
                           "ranking_field": field in RANKING_FIELDS})
field_df = pd.DataFrame(field_rows)
field_df

,dataset,field,accepted,rejected,null,ranking_field
0,synthetic,funding_type,89,59,0,True
1,synthetic,amount_total,137,1,10,False
2,synthetic,currency,132,9,7,False
3,synthetic,deadline,139,1,8,False
4,synthetic,target_applicants,116,21,11,True
5,synthetic,focus_area,99,45,4,True
6,synthetic,geography,96,33,19,True
7,tracker,funding_type,28,9,0,True
8,tracker,amount_total,24,5,8,False
9,tracker,currency,20,7,10,False


In [7]:
# Why fields were rejected (first words of each reason), and grants with no usable JSON.
reasons = Counter((row["dataset"], field, reason.split(":")[0])
                  for row in extraction_rows if row.get("rejected")
                  for field, reason in row["rejected"].items())
display(pd.DataFrame([{"dataset": d, "field": f, "reason": r, "count": n} for (d, f, r), n in reasons.most_common()]))
extraction_df[~extraction_df["status"].str.startswith("ok")][["dataset", "source_row", "program", "status", "error"]]

,dataset,field,reason,count
0,synthetic,focus_area,not copied verbatim from the source,40
1,synthetic,funding_type,not copied verbatim from the source,32
2,synthetic,geography,not copied verbatim from the source,28
3,synthetic,funding_type,not found in type/program,27
4,synthetic,target_applicants,not copied verbatim from the source,14
5,tracker,funding_type,not copied verbatim from the source,7
6,synthetic,currency,not copied verbatim from the source,7
7,tracker,currency,not copied verbatim from the source,6
8,tracker,amount_total,not copied verbatim from the source,5
9,tracker,geography,not copied verbatim from the source,4


,dataset,source_row,program,status,error
114,synthetic,79,Prairie Roots Initiative,failed,DeepSeek response did not contain a JSON object
125,synthetic,90,Sparrow Futures Fund,failed,DeepSeek response did not contain a JSON object


In [8]:
# Validated tracker values (all fields; only RANKING_FIELDS feed the ranking).
pd.DataFrame([{"source_row": row, **tracker_details[row]} for row in sorted(tracker_details)])

,source_row,funding_type,amount_total,currency,deadline,target_applicants,focus_area,geography
0,2,Grant + support,"€75,000 non-dilutive + mentoring/coaching",€,2026-09-16,Woman founder/co-founder in top management; women collectively ≥25% ownershi...,deep-tech; TRL/technical maturity requirements,Armenia; deep-tech; TRL/technical maturity requirements
1,3,Prize / cash awards,"€100,000",€,2026-09-17,Women-led; international; seed-stage,Women-led; international; seed-stage,Women-led; international; seed-stage
2,4,Grant,"$5,000",$,2026-09-18,Woman entrepreneur; operating business; newsletter subscription; application...,Woman entrepreneur; operating business; newsletter subscription; application...,None
3,5,Grant,"$10,000",None,2026-09-30,None,None,None
4,6,Grant,"$10,000",$,2026-09-30,Women-owned business,Women-owned business,None
5,7,Grant + support,"$2,500",$,2026-09-30,Female entrepreneur; growth potential; impact/leadership/service criteria,Female entrepreneur; growth potential; impact/leadership/service criteria,Female entrepreneur; growth potential; impact/leadership/service criteria
6,8,None,"$30,000",$,2026-09-30,U.S.-based active for-profit; ≥6 months operating; business bank account; ≥$...,U.S.-based active for-profit; ≥6 months operating; business bank account; ≥$...,U.S.-based active for-profit; ≥6 months operating; business bank account; ≥$...
7,9,Microgrant,$500,None,2026-09-30,"Women entrepreneurs; proposed activity must benefit women, girls and/or fami...",Women entrepreneurs,None
8,10,GTIA Grants,"$25,000–$50,000",$,2026-10-02,Nonprofit,Nonprofit,Nonprofit
9,11,Grant + fellowship,"$3,000–$10,000",$,2026-11-01,Early-stage women/non-binary founders,Early-stage women/non-binary founders,Early-stage women/non-binary founders


## 4. Ranking comparison

`semantic + DeepSeek` adds one ranking chunk per accepted ranking field (`field: value`), exactly as
`src.recommend --deepseek-extract` does. Everything else is the baseline pipeline.

In [9]:
def chunks_from(details_by_row):
    out = {}
    for row, details in details_by_row.items():
        chunks = [f"{f}: {details[f]}" for f in RANKING_FIELDS if details[f] is not None]
        if chunks:
            out[row] = chunks
    return out


EMBEDDER = SemanticEncoder()  # shared so every pipeline uses the identical MiniLM model


def evaluate(pipeline, profiles, labels_by_profile, relevant_by_profile, as_of, threshold):
    per_profile = []
    for pid, profile in profiles:
        top = pipeline.recommend(profile, as_of=as_of, top_k=K, min_similarity=threshold)
        relevant = relevant_by_profile[pid]
        hits = sum(item.source_row in relevant for item in top)
        per_profile.append({
            "profile_id": pid, "hits": hits, "precision_at_5": hits / K,
            "recall_at_5": hits / len(relevant) if relevant else 0.0,
            "top_five": [{"rank": i + 1, "source_row": item.source_row, "program": item.program,
                          "label": labels_by_profile[pid].get(item.source_row), "relevant": item.source_row in relevant,
                          "ranking_score": round(item.score, 2), "cosine_similarity": round(item.cosine_similarity, 4)}
                         for i, item in enumerate(top)],
        })
    return per_profile


tracker_chunks, syn_chunks = chunks_from(tracker_details), chunks_from(syn_details)
tracker_inputs = ([(0, TRACKER_PROFILE)], {0: tracker_labels}, {0: tracker_relevant}, TRACKER_DATE)
syn_profile_list = [(int(p.id), OrganizationProfile(p.description, p.attributes)) for p in syn_profiles.itertuples()]
syn_inputs = (syn_profile_list, syn_labels,
              {pid: {r for r, l in labels.items() if l == "match"} for pid, labels in syn_labels.items()}, SYN_DATE)

runs = {
    ("tracker", "baseline semantic"): (Pipeline(tracker_grants, embedder=EMBEDDER, checker=tracker_checker, store=MemoryStore()),
                                       tracker_inputs, DEFAULT_MIN_SIMILARITY),
    ("tracker", "semantic + DeepSeek"): (Pipeline(tracker_grants, embedder=EMBEDDER, checker=tracker_checker, store=MemoryStore(),
                                                  additional_chunks=tracker_chunks, extracted_details=tracker_details),
                                         tracker_inputs, DEFAULT_MIN_SIMILARITY),
    ("tracker", "keyword (reference)"): (Pipeline(tracker_grants, checker=tracker_checker, embedder=KeywordEmbedder(),
                                                  store=MemoryStore(), chunk_weight=0), tracker_inputs, 1e-12),
    ("synthetic", "baseline semantic"): (Pipeline(syn_grants, embedder=EMBEDDER, checker=syn_checker, store=MemoryStore()),
                                         syn_inputs, DEFAULT_MIN_SIMILARITY),
    ("synthetic", "semantic + DeepSeek"): (Pipeline(syn_grants, embedder=EMBEDDER, checker=syn_checker, store=MemoryStore(),
                                                    additional_chunks=syn_chunks, extracted_details=syn_details),
                                           syn_inputs, DEFAULT_MIN_SIMILARITY),
    ("synthetic", "keyword (reference)"): (Pipeline(syn_grants, checker=syn_checker, embedder=KeywordEmbedder(),
                                                    store=MemoryStore(), chunk_weight=0), syn_inputs, 1e-12),
}
per_profile = {key: evaluate(pipe, *inputs, threshold) for key, (pipe, inputs, threshold) in runs.items()}

summary = pd.DataFrame([
    {"dataset": d, "method": m, "profiles": len(rows),
     "Precision@5": round(np.mean([r["precision_at_5"] for r in rows]), 3),
     "Recall@5": round(np.mean([r["recall_at_5"] for r in rows]), 3)}
    for (d, m), rows in per_profile.items()
])
summary

,dataset,method,profiles,Precision@5,Recall@5
0,tracker,baseline semantic,1,0.600,0.375
1,tracker,semantic + DeepSeek,1,0.600,0.375
2,tracker,keyword (reference),1,0.400,0.250
3,synthetic,baseline semantic,200,0.470,0.377
4,synthetic,semantic + DeepSeek,200,0.455,0.365
5,synthetic,keyword (reference),200,0.369,0.291


In [10]:
for method in ("baseline semantic", "semantic + DeepSeek", "keyword (reference)"):
    print(f"\nTracker — {method}")
    display(pd.DataFrame(per_profile[("tracker", method)][0]["top_five"]).set_index("rank"))


Tracker — baseline semantic


,source_row,program,label,relevant,ranking_score,cosine_similarity
rank,,,,,,
1,19,AI for Good Innovation Factory,Moderate,False,24.22,0.2444
2,35,NetHope AI Accelerator,Strong,True,24.20,0.2379
3,17,NVIDIA Inception,Strong,True,23.51,0.2000
4,21,Google for Startups Women Founders Fund,Excellent if a matching round opens,False,21.00,0.2007
5,11,EchoHer Founder Fellowship,Excellent,True,20.58,0.2155



Tracker — semantic + DeepSeek


,source_row,program,label,relevant,ranking_score,cosine_similarity
rank,,,,,,
1,19,AI for Good Innovation Factory,Moderate,False,24.22,0.2444
2,35,NetHope AI Accelerator,Strong,True,24.20,0.2379
3,17,NVIDIA Inception,Strong,True,23.51,0.2000
4,21,Google for Startups Women Founders Fund,Excellent if a matching round opens,False,21.00,0.2007
5,11,EchoHer Founder Fellowship,Excellent,True,20.58,0.2155



Tracker — keyword (reference)


,source_row,program,label,relevant,ranking_score,cosine_similarity
rank,,,,,,
1,12,100x Impact Accelerator 2027/28,Strong,True,8.89,0.0889
2,19,AI for Good Innovation Factory,Moderate,False,6.82,0.0682
3,18,UNICEF Venture Fund,Moderate,False,5.63,0.0563
4,33,Llama Impact Grants,Strong,True,5.11,0.0511
5,22,Google.org GenAI / AI Social Impact Accelerator,Excellent if matching call opens,False,4.97,0.0497


In [11]:
# Sanity check against the recorded tracker baseline (2026-10-28) and `make synthetic` (47.0% / 37.7%).
checks = {("tracker", "baseline semantic"): 0.60, ("tracker", "keyword (reference)"): 0.40,
          ("synthetic", "baseline semantic"): 0.470, ("synthetic", "keyword (reference)"): 0.369}
for key, expected in checks.items():
    got = np.mean([r["precision_at_5"] for r in per_profile[key]])
    print(f"{key}: P@5 {got:.3f} vs recorded {expected:.3f} — {'matches' if abs(got - expected) < 0.0015 else 'DIFFERS'}")

('tracker', 'baseline semantic'): P@5 0.600 vs recorded 0.600 — matches
('tracker', 'keyword (reference)'): P@5 0.400 vs recorded 0.400 — matches
('synthetic', 'baseline semantic'): P@5 0.470 vs recorded 0.470 — matches
('synthetic', 'keyword (reference)'): P@5 0.369 vs recorded 0.369 — matches


In [12]:
# Synthetic: paired per-profile comparison (same 200 profiles).
base = {r["profile_id"]: r["hits"] for r in per_profile[("synthetic", "baseline semantic")]}
ds = {r["profile_id"]: r["hits"] for r in per_profile[("synthetic", "semantic + DeepSeek")]}
diff = np.array([ds[p] - base[p] for p in base])
better, worse = int((diff > 0).sum()), int((diff < 0).sum())
from math import comb
n = better + worse
sign_p = (sum(comb(n, k) for k in range(0, min(better, worse) + 1)) * 2 / 2 ** n) if n else 1.0
SYN_PAIRED = {"profiles_better": better, "profiles_worse": worse, "profiles_same": int((diff == 0).sum()),
              "mean_hit_change": float(diff.mean()), "sign_test_p_two_sided": min(1.0, sign_p)}
SYN_PAIRED

{'profiles_better': 8,
 'profiles_worse': 22,
 'profiles_same': 170,
 'mean_hit_change': -0.075,
 'sign_test_p_two_sided': 0.016124801710247993}

## 5. Result and saved output

In [13]:
def metric(d, m, k):
    return float(np.mean([r[k] for r in per_profile[(d, m)]]))

lines = []
for d in ("tracker", "synthetic"):
    b, x = metric(d, "baseline semantic", "precision_at_5"), metric(d, "semantic + DeepSeek", "precision_at_5")
    br, xr = metric(d, "baseline semantic", "recall_at_5"), metric(d, "semantic + DeepSeek", "recall_at_5")
    change = "higher" if x > b + 1e-9 else "lower" if x < b - 1e-9 else "unchanged"
    lines.append(f"{d}: baseline P@5 {b:.1%} / Recall {br:.1%}; + DeepSeek P@5 {x:.1%} / Recall {xr:.1%} ({change})")
ok_tracker = sum(1 for r in tracker_rows if r["status"].startswith("ok"))
ok_syn = sum(1 for r in syn_rows if r["status"].startswith("ok"))
lines.append(f"Usable extractions: tracker {ok_tracker}/{len(tracker_rows)}, synthetic {ok_syn}/{len(syn_rows)}; "
             f"synthetic paired: {SYN_PAIRED['profiles_better']} profiles better, {SYN_PAIRED['profiles_worse']} worse "
             f"(sign test p={SYN_PAIRED['sign_test_p_two_sided']:.3g}).")
conclusion = "\n".join(lines)
print(conclusion)

RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
RESULTS_PATH.write_text(json.dumps({
    "model": MODEL_NAME, "extraction_version": EXTRACTION_VERSION, "environment": ENVIRONMENT,
    "settings": {"top_k": K, "min_similarity": DEFAULT_MIN_SIMILARITY, "chunk_weight": DEFAULT_CHUNK_WEIGHT,
                 "embedder": "sentence-transformers/all-MiniLM-L6-v2", "ranking_fields": list(RANKING_FIELDS)},
    "tracker": {"evaluation_date": TRACKER_DATE.isoformat(), "profile": TRACKER_PROFILE.description,
                "attributes": TRACKER_PROFILE.attributes, "relevant_rows": sorted(tracker_relevant),
                "label_caveat": "Tracker Strong/Excellent Fit ratings; not independently verified suitability."},
    "synthetic": {"reference_date": SYN_DATE.isoformat(), "profiles": len(syn_profile_list), "paired": SYN_PAIRED,
                  "label_caveat": "Rule-generated labels; not real-world accuracy evidence."},
    "summary": summary.to_dict("records"),
    "tracker_top_five": {m: per_profile[("tracker", m)][0]["top_five"] for (d, m) in per_profile if d == "tracker"},
    "field_outcomes": field_df.to_dict("records"),
    "extraction": extraction_rows,
    "conclusion": conclusion,
}, ensure_ascii=False, indent=2, default=str) + "\n", encoding="utf-8")
print(f"Saved {RESULTS_PATH.relative_to(ROOT)}")

tracker: baseline P@5 60.0% / Recall 37.5%; + DeepSeek P@5 60.0% / Recall 37.5% (unchanged)
synthetic: baseline P@5 47.0% / Recall 37.7%; + DeepSeek P@5 45.5% / Recall 36.5% (lower)
Usable extractions: tracker 37/37, synthetic 148/150; synthetic paired: 8 profiles better, 22 worse (sign test p=0.0161).
Saved notebooks/results/deepseek_ranking_evaluation.json
